In [1]:
import os
import joblib
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# ==========================================
# 1. DATA INGESTION & CLEANING (GLOBAL UNDER-SAMPLING)
# ==========================================

dataset_path = '../datasets/CIC-UNSW-NB15/CICFlowMeter_out.csv'
df = pd.read_csv(dataset_path, encoding='latin-1', low_memory=False)
df.columns = df.columns.str.strip()

# Drop identifiers, IPs, and Ports to prevent data leakage
cols_to_drop = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# Sanitize
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df['Label'] = df['Label'].str.upper().str.strip()

# --- THE NEW UNDERSAMPLING STEP ---
print(f"Original dataset shape: {df.shape}")

# Separate Benign and Attacks
df_benign = df[df['Label'] == 'BENIGN']
df_attacks = df[df['Label'] != 'BENIGN']

# Sample 800,000 Benign rows (or keep all if less than 800k)
sample_size = min(400000, len(df_benign))
df_benign_sampled = df_benign.sample(n=sample_size, random_state=42)

# Combine and shuffle
master_df = pd.concat([df_benign_sampled, df_attacks]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Undersampled dataset shape: {master_df.shape}")
print(f"New Ratio (Benign:Attack): {len(df_benign_sampled) / len(df_attacks):.1f} : 1")
# ----------------------------------

X = master_df.drop(columns=['Label'])
y_multi = master_df['Label']

# Encoding Multi Class labels (stage 2)
le = LabelEncoder()
y_multi_encoded = le.fit_transform(y_multi)

# Assigning Binary Labels (Stage 1) | 0 = BENIGN ; 1 = ATTACK
y_binary = (y_multi != 'BENIGN').astype(int)

# Splitting data 
X_train, X_test, y_multi_train, y_multi_test, y_bin_train, y_bin_test = train_test_split(
    X, y_multi_encoded, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_multi_encoded
)

# Encoding Categorical features
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

joblib.dump(list(X_train.columns), '../Models/unsw/train_columns.joblib')

print("✅ UNSW train_columns.joblib saved successfully!")

Original dataset shape: (3540241, 78)
Undersampled dataset shape: (489583, 78)
New Ratio (Benign:Attack): 4.5 : 1
X_train shape: (391666, 77)
X_test shape: (97917, 77)
✅ UNSW train_columns.joblib saved successfully!


In [ ]:
import lightgbm as lgb
# ==========================================
# STAGE 1: BINARY TRIAGE (Benign vs Attack)
# ==========================================
print("\n" + "="*50)
print("STAGE 1: FEATURE SELECTION & RETRAINING")
print("="*50)

# 1. Train a quick baseline model to find the best features
base_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
base_rf.fit(X_train, y_bin_train)

# 2. Extract and sort feature importances
importances = base_rf.feature_importances_
feature_names = X_train.columns
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False)

# 3. Keep ONLY the top 30 features
top_30_features = fi_df.head(30)['Feature'].tolist()
print(f"Top 5 Features: {top_30_features[:5]}")

X_train_fs = X_train[top_30_features]
X_test_fs = X_test[top_30_features]

# 4. Retrain Stage 1 on the reduced dataset
stage1_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

stage1_model.fit(X_train_fs, y_bin_train)

# 5. Predict with a standard 0.5 threshold (Feature selection usually fixes the boundary)
y_probs = stage1_model.predict_proba(X_test_fs)[:, 1]
stage1_preds_custom = (y_probs >= 0.7).astype(int)

print("\n--- Stage 1 with Feature Selection ---")
print(classification_report(y_bin_test, stage1_preds_custom, target_names=['BENIGN', 'ATTACK']))

cm_bin = confusion_matrix(y_bin_test, stage1_preds_custom)
tn, fp, fn, tp = cm_bin.ravel()
print(f"False Positives (False Alarms): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")

In [ ]:
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import numpy as np
from sklearn.metrics import classification_report

# ==========================================
# STAGE 2: MULTI-CLASS DETECTIVE
# ==========================================
print("\n" + "="*50)
print("STAGE 2: TRAINING MULTI-CLASS DETECTIVE MODEL")
print("="*50)

# 1. Filter out BENIGN from training data
benign_idx = le.transform(['BENIGN'])[0]
attack_mask_train = (y_multi_train != benign_idx)

# USE THE FULL FEATURE SET (X_train, not X_train_fs)
X_train_attacks = X_train[attack_mask_train]
y_train_attacks = y_multi_train[attack_mask_train]

# 2. Apply SMOTE to balance the attack classes
# This creates synthetic data for rare classes like Worms and Backdoor
print("Applying SMOTE to balance attack classes...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_attacks, y_train_attacks)

# 3. Train LightGBM (Much better for multi-class)
print("Fitting the Stage 2 LightGBM model...")
stage2_model = lgb.LGBMClassifier(
    n_estimators=250,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

stage2_model.fit(X_train_smote, y_train_smote)

# 4. Evaluate Stage 2 ONLY on true attacks from the test set
attack_mask_test = (y_bin_test == 1)
X_test_attacks = X_test[attack_mask_test] # Use full features
y_test_attacks = y_multi_test[attack_mask_test]

stage2_preds_attacks = stage2_model.predict(X_test_attacks)

print("\n--- Stage 2 Standalone Report (Attacks Only) ---")
# Drop BENIGN from the report
print(classification_report(
    y_test_attacks, 
    stage2_preds_attacks, 
    labels=range(1, len(le.classes_)), 
    target_names=le.classes_[1:], 
    zero_division=0
))

In [ ]:
print("\n" + "="*50)
print("COMBINED CHAINED PIPELINE EVALUATION")
print("="*50)

# 1. Stage 1 Predicts (using Top 30)
y_probs_s1 = stage1_model.predict_proba(X_test_fs)[:, 1]
stage1_preds = (y_probs_s1 >= 0.5).astype(int)

# 2. Stage 2 Predicts (using Full Features and LightGBM)
stage2_preds_all = stage2_model.predict(X_test) # Use your LightGBM model here
stage2_preds_labels = le.inverse_transform(stage2_preds_all)

# 3. Chain them
final_prediction = np.where(stage1_preds == 0, 'BENIGN', stage2_preds_labels)

# 4. Final Report
print(classification_report(
    le.inverse_transform(y_multi_test),
    final_prediction,
    zero_division=0
))